In [ ]:
import numpy as np    #Importing required libraries
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
import os


(imgtrain, _), (imgtest, _) = mnist.load_data() #Getting test and training images
images = np.concatenate([imgtrain, imgtest], axis=0)  # total 70,000 images after merging training and testing images


trainmasks = np.load("/content/mnistforegrounddataset/trainmasks.npy") #Path to foreground masks created earlier
testmasks  = np.load("/content/mnistforegrounddataset/testmasks.npy")
masks = np.concatenate([trainmasks, testmasks], axis=0) #Similarly merging both test and training masks to get 70000 images


print("Images shape:", images.shape) #Getting size of images and masks
print("Masks shape:", masks.shape)


def creategridimage(imgs, masks): #Function to create 2 cross 2 grid

    top = np.hstack((imgs[0], imgs[1]))    #Top row of the grid
    bottom = np.hstack((imgs[2], imgs[3])) # bottom row of the grid
    gridimage = np.vstack((top, bottom)) #Combining top and bottom rows to get grid of images


    topm = np.hstack((masks[0], masks[1])) #similarly getting grid of masks
    bottomm = np.hstack((masks[2], masks[3]))
    gridmask = np.vstack((topm, bottomm))


    gridmask = (gridmask > 0).astype(np.uint8) #Ensuring that mask grid is binary


    return gridimage, gridmask


numsamples = 250000   #Ensuring that there are total 250000 images in the dataset
combinedimages = [] #Creating array for combined images and masks
combinedmasks = []


for _ in range(numsamples):
    index = np.random.choice(len(images), 4, replace=False) #Getting 4 different and random indexes of array of images
    imgs = [images[i] for i in index] #Creating array of images from those 4 values of indexes
    msks = [masks[i] for i in index]


    imggrid, maskgrid = creategridimage(imgs, msks) #Getting grid of images and masks
    combinedimages.append(imggrid) #Storing results in an array made earlier
    combinedmasks.append(maskgrid)


combinedimages = np.array(combinedimages, dtype=np.uint8) #converting to numpy arrays for storage
combinedmasks = np.array(combinedmasks, dtype=np.uint8)


print("\nGenerated dataset shapes:")  #Printing sizes of images ,  masks and grids
print("Images:", combinedimages.shape)
print("Masks:", combinedmasks.shape)
print("Each grid image shape:", combinedimages[0].shape)
print("Each grid mask shape:", combinedmasks[0].shape)
print("Unique values in masks:", np.unique(combinedmasks))


os.makedirs("mnistsemanticsegdataset", exist_ok=True) #Saving dataset
np.save("mnistsemanticsegdataset/gridimages.npy", combinedimages)
np.save("mnistsemanticsegdataset/gridmasks.npy", combinedmasks)


#Plottimg sample results
n = 3
plt.figure(figsize=(10, 6))
for i in range(n):
    plt.subplot(2, n, i + 1)
    plt.imshow(combinedimages[i], cmap='gray')
    plt.title("Grid of Image")
    plt.axis('off')


    plt.subplot(2, n, i + 1 + n)
    plt.imshow(combinedmasks[i], cmap='gray')
    plt.title("Grid of Mask")
    plt.axis('off')


plt.tight_layout() #Plotting results
plt.show()


totalimgs = combinedimages.shape[0] #Getting total number of images in the dataset
totalmasks = combinedmasks.shape[0]
uniquevals = np.unique(combinedmasks)
isbinary = np.array_equal(uniquevals, [0, 1]) #ensuring that masks are binary


print(f"\n Total combined images: {totalimgs}")
print(f" Total combined masks: {totalmasks}")
print(f" Unique values in masks: {uniquevals}")
print(f" Masks are binary: {isbinary}")
